In [1]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
# Locate project root

PROJECT_ROOT = Path.cwd()

while not (
    (PROJECT_ROOT / "data" / "train").exists()
    and (PROJECT_ROOT / "output").exists()
):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError(
            "Could not locate amazon-ml-challenge project root."
        )
    PROJECT_ROOT = PROJECT_ROOT.parent

OUTPUT_DIR = PROJECT_ROOT / "output"
FEATURE_DIR = OUTPUT_DIR / "training_features_v1"

features = np.load(
    FEATURE_DIR / "features.npy",
    mmap_mode="r"
)

label = np.load(
    FEATURE_DIR / "labels.npy"
)

s1_idx = np.load(
    FEATURE_DIR / "s1_idx.npy"
)

target_num = np.load(
    FEATURE_DIR / "target_num.npy"
)

source = np.load(
    FEATURE_DIR / "source.npy"
)

with open(
    FEATURE_DIR / "feature_names.pkl",
    "rb"
) as f:
    feature_names = pickle.load(f)

print("Feature matrix:", features.shape)
print("Labels:", label.shape)
print("Features:", feature_names)

Feature matrix: (5729367, 15)
Labels: (5729367,)
Features: ['name_exact', 'name_ratio', 'name_token_ratio', 'name_token_jaccard', 'name_length_ratio', 'name_digit_overlap', 'address_exact', 'address_ratio', 'address_token_ratio', 'address_token_jaccard', 'address_length_ratio', 'address_digit_overlap', 'country_same', 'name_present_both', 'address_present_both']


In [3]:
# Add legitimate source information:
# 0 = S2
# 1 = S3

source_feature = (
    (source == 3)
    .astype(np.float32)
    .reshape(-1, 1)
)

X = np.empty(
    (features.shape[0], features.shape[1] + 1),
    dtype=np.float32
)

X[:, :-1] = features
X[:, -1] = source_feature[:, 0]

MODEL_FEATURE_NAMES = feature_names + [
    "source_is_s3"
]

print("Model matrix:", X.shape)
print("Features:", MODEL_FEATURE_NAMES)

Model matrix: (5729367, 16)
Features: ['name_exact', 'name_ratio', 'name_token_ratio', 'name_token_jaccard', 'name_length_ratio', 'name_digit_overlap', 'address_exact', 'address_ratio', 'address_token_ratio', 'address_token_jaccard', 'address_length_ratio', 'address_digit_overlap', 'country_same', 'name_present_both', 'address_present_both', 'source_is_s3']


In [4]:
positive_count = int((label == 1).sum())
negative_count = int((label == 0).sum())

print("Positive:", positive_count)
print("Negative:", negative_count)

print(
    "Positive fraction:",
    f"{positive_count / len(label):.4%}"
)

print(
    "Negative fraction:",
    f"{negative_count / len(label):.4%}"
)

Positive: 3643468
Negative: 2085899
Positive fraction: 63.5929%
Negative fraction: 36.4071%


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

In [6]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print("Scaled matrix:", X_scaled.shape)

Scaled matrix: (5729367, 16)


In [7]:
linear_model = LogisticRegression(
    solver="saga",
    penalty="l2",
    C=1.0,
    max_iter=40,
    tol=1e-3,
    n_jobs=-1,
    verbose=1
)

linear_model.fit(
    X_scaled,
    label
)

print("Logistic model training complete.")

/Users/bhavikesh/Documents/Hackathon/Amazon_ml/amazon-ml-challenge/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/bhavikesh/Documents/Hackathon/Amazon_ml/amazon-ml-challenge/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Epoch 1, change: 1
Epoch 2, change: 0.4509435
Epoch 3, change: 0.14624125
Epoch 4, change: 0.050606672
Epoch 5, change: 0.052138269
Epoch 6, change: 0.022204222
Epoch 7, change: 0.0045718341
Epoch 8, change: 0.051401634
Epoch 9, change: 0.0087482575
Epoch 10, change: 0.0011396965
convergence after 11 epochs took 31 seconds
Logistic model training complete.


In [8]:
from xgboost import XGBClassifier

In [9]:
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,

    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.9,

    reg_alpha=0.0,
    reg_lambda=5.0,

    objective="binary:logistic",
    eval_metric="logloss",

    tree_method="hist",
    n_jobs=-1,

    random_state=42
)

xgb_model.fit(
    X,
    label,
    verbose=True
)

print("XGBoost training complete.")

XGBoost training complete.


In [10]:
from lightgbm import LGBMClassifier

In [11]:
lgb_model = LGBMClassifier(
    n_estimators=250,
    learning_rate=0.05,

    num_leaves=31,
    max_depth=-1,

    min_child_samples=100,

    subsample=0.8,
    colsample_bytree=0.9,

    reg_alpha=0.0,
    reg_lambda=5.0,

    objective="binary",
    random_state=42,

    n_jobs=-1
)

lgb_model.fit(
    X,
    label
)

print("LightGBM training complete.")

[LightGBM] [Info] Number of positive: 3643468, number of negative: 2085899
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.024829 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2
[LightGBM] [Info] Number of data points in the train set: 5729367, number of used features: 1
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.635929 -> initscore=0.557736
[LightGBM] [Info] Start training from score 0.557736
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

In [12]:
MODEL_DIR = OUTPUT_DIR / "models_v1"
MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

with open(
    MODEL_DIR / "logistic_model.pkl",
    "wb"
) as f:
    pickle.dump(
        linear_model,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )

with open(
    MODEL_DIR / "logistic_scaler.pkl",
    "wb"
) as f:
    pickle.dump(
        scaler,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )

with open(
    MODEL_DIR / "xgboost_model.pkl",
    "wb"
) as f:
    pickle.dump(
        xgb_model,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )

with open(
    MODEL_DIR / "lightgbm_model.pkl",
    "wb"
) as f:
    pickle.dump(
        lgb_model,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )

with open(
    MODEL_DIR / "model_feature_names.pkl",
    "wb"
) as f:
    pickle.dump(
        MODEL_FEATURE_NAMES,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )

print("Models saved to:")
print(MODEL_DIR)

Models saved to:
/Users/bhavikesh/Documents/Hackathon/Amazon_ml/amazon-ml-challenge/output/models_v1
